In [ ]:
import pandas as pd
import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import pipeline
from sklearn import impute
from sklearn import compose

# Pipelines

In [ ]:
# Datos
# https://raw.githubusercontent.com/bencmbit/datasets/master/AER_credit_card_data.csv

# card: 1 si la aplicación a la tarjeta de crédito fue aceptada, 0 si no lo fue
# reports: Número de reportes malos importantes
# age: Edad
# income: Ingresos anuales (divido 10000)
# share: Gastos mensuales de tarjeta de crédito / ingreso anual
# expenditure: Gasto promedio mensual en tarjeta de crédito
# owner: 1 si es propietario, 0 si alquila
# selfempl: 1 si es trabajador por cuenta propia, 0 si no lo es
# dependents: 1 + número de gente que depende de esta persona
# months: Meses viviendo en la dirección actual
# majorcards: Cantidad de tarjetas de crédito importantes que tiene
# active: Número de cuentas de crédito activas

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/bencmbit/datasets/master/AER_credit_card_data.csv")
df = df.replace({"yes": 1, "no": 0})
df.head()

/tmp/ipykernel_36638/1022621242.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({"yes": 1, "no": 0})


,card,reports,age,income,share,expenditure,owner,selfemp,dependents,months,majorcards,active
0,1,0,37.66667,4.5200,0.033270,124.983300,1,0,3,54,1,12
1,1,0,33.25000,2.4200,0.005217,9.854167,0,0,3,34,1,13
2,1,0,33.66667,4.5000,0.004156,15.000000,1,0,4,58,1,5
3,1,0,30.50000,2.5400,0.065214,137.869200,0,0,0,25,1,7
4,1,0,32.16667,9.7867,0.067051,546.503300,1,0,2,64,1,5


In [ ]:
X = df[df.columns.drop(["card", "expenditure", "share", "active", "majorcards"])]
y = df["card"]

clf = sk.ensemble.RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

scores = sk.model_selection.cross_val_score(clf, X, y, cv=5, scoring='accuracy')
print("accuracy: %f" % scores.mean())

accuracy: 0.833961


# Pipelines versión 1: Hacer limpieza en folds

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/bencmbit/datasets/master/AER_credit_card_data.csv")
df = df.replace({"yes": 1, "no": 0})
df.head()

/tmp/ipykernel_36638/1022621242.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({"yes": 1, "no": 0})


,card,reports,age,income,share,expenditure,owner,selfemp,dependents,months,majorcards,active
0,1,0,37.66667,4.5200,0.033270,124.983300,1,0,3,54,1,12
1,1,0,33.25000,2.4200,0.005217,9.854167,0,0,3,34,1,13
2,1,0,33.66667,4.5000,0.004156,15.000000,1,0,4,58,1,5
3,1,0,30.50000,2.5400,0.065214,137.869200,0,0,0,25,1,7
4,1,0,32.16667,9.7867,0.067051,546.503300,1,0,2,64,1,5


In [ ]:
X = df[["age"]]
p = sk.preprocessing.MinMaxScaler()
p.fit_transform(X) # p.fit(X) + p.transform(X)

array([[0.45000004],
       [0.397     ],
       [0.40200004],
       ...,
       [0.48499996],
       [0.39199996],
       [0.577     ]])

In [ ]:
X = df[df.columns.drop(["card", "expenditure", "share", "active", "majorcards"])]
y = df["card"]

scores_train = []
scores_test = []

clf = sk.ensemble.RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

p = sk.preprocessing.MinMaxScaler()

kf = sk.model_selection.KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_index, test_index) in enumerate(kf.split(X, y)):

    # Partimos el fold en entrenamiento y prueba...
    X_train, X_test, y_train, y_test = X.iloc[train_index], X.iloc[test_index], y.iloc[train_index], y.iloc[test_index]

    # Limpieza de datos
    p.fit(X_train)
    X_train = p.transform(X_train)
    X_test = p.transform(X_test)

    # Entrenamos el modelo en entramiento
    clf.fit(X_train, y_train)

    # Predecimos en train
    y_pred = clf.predict(X_train)

    # Medimos la performance de la predicción en entramiento
    score_train = sk.metrics.accuracy_score(y_train, y_pred)
    scores_train.append(score_train)

    # Predecimos en test
    y_pred = clf.predict(X_test)

    # Medimos la performance de la predicción en prueba
    score_test = sk.metrics.accuracy_score(y_test, y_pred)
    scores_test.append(score_test)

    print(f"{fold=}, {score_train=} {score_test=}")

print(f"scores_train mean={pd.Series(scores_train).mean()}, std={pd.Series(scores_train).std()}")
print(f"scores_test  mean={pd.Series(scores_test).mean()}, std={pd.Series(scores_test).std()}")

fold=0, score_train=0.9990521327014218 score_test=0.821969696969697
fold=1, score_train=1.0 score_test=0.8371212121212122
fold=2, score_train=1.0 score_test=0.7992424242424242
fold=3, score_train=1.0 score_test=0.8333333333333334
fold=4, score_train=1.0 score_test=0.844106463878327
scores_train mean=0.9998104265402844, std=0.00042389914265399294
scores_test  mean=0.8271546261089988, std=0.017541725732382378


# Pipelines versión 2: Usar pipelines de sklearn

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/bencmbit/datasets/master/AER_credit_card_data.csv")
df = df.replace({"yes": 1, "no": 0})
df.head()

/tmp/ipykernel_36638/1022621242.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace({"yes": 1, "no": 0})


,card,reports,age,income,share,expenditure,owner,selfemp,dependents,months,majorcards,active
0,1,0,37.66667,4.5200,0.033270,124.983300,1,0,3,54,1,12
1,1,0,33.25000,2.4200,0.005217,9.854167,0,0,3,34,1,13
2,1,0,33.66667,4.5000,0.004156,15.000000,1,0,4,58,1,5
3,1,0,30.50000,2.5400,0.065214,137.869200,0,0,0,25,1,7
4,1,0,32.16667,9.7867,0.067051,546.503300,1,0,2,64,1,5


In [ ]:
from sklearn import ensemble
from sklearn import pipeline

X = df[df.columns.drop(["card", "expenditure", "share", "active", "majorcards"])]
y = df["card"]

pipe = sk.pipeline.Pipeline(steps=[
        ("mms", sk.preprocessing.MinMaxScaler()),
        ("si", sk.impute.SimpleImputer(strategy='median', add_indicator=True)),
        ("clf", sk.ensemble.RandomForestClassifier(n_estimators=100, random_state=42)),
])

scores = sk.model_selection.cross_val_score(pipe, X, y, scoring="accuracy", cv=5, n_jobs=-1)
print("scores mean:", scores.mean())

In [ ]:
from sklearn import ensemble
from sklearn import pipeline

X = df[df.columns.drop(["card", "expenditure", "share", "active", "majorcards"])]
y = df["card"]

ct = sk.compose.ColumnTransformer([
    ("imp_reports", sk.impute.SimpleImputer(strategy="median", add_indicator=True), ["reports"]),
    ("imp_age", sk.impute.SimpleImputer(strategy="median", add_indicator=True), ["age"]),
    ("imp_dependents", sk.pipeline.make_pipeline(sk.impute.SimpleImputer(strategy="median", add_indicator=True), sk.preprocessing.MinMaxScaler()), ["dependents"])
], remainder="drop", n_jobs=-1)

pipe = sk.pipeline.Pipeline(steps=[
    ("ct", ct),
    ("clf", sk.ensemble.RandomForestClassifier(n_estimators=100, random_state=42)),
])

scores = sk.model_selection.cross_val_score(pipe, X, y, scoring="accuracy", cv=10, n_jobs=-1)
print("scores mean:", scores.mean())

scores mean: 0.762676382142031


# Pipelines versión 3: Mis propios pipelines 👓

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class AgeMaxer(BaseEstimator, TransformerMixin):
    def __init__(self, max_age=100):
        print("Inicializo el transformador...")
        self.max_age = max_age

    def fit(self, X, y=None): # X es train...
        self.mean_age = round(X['age'].mean())
        print("Guardando la media de la edad...", self.mean_age)
        return self

    def transform(self, X): # X puede ser train o test
        print ("Reemplazando valores de edad imposibles con el promedio...")
        X.loc[(X['age'] < 0) | (X['age'] > self.max_age), "age"] = self.mean_age

        return X

In [ ]:
X = df[df.columns.drop(["card", "expenditure", "share", "active", "majorcards"])]
y = df["card"]

p = AgeMaxer(30)
p.fit(X)
p.transform(X)

Inicializo el transformador...
Guardando la media de la edad... 33
Reemplazando valores de edad imposibles con el promedio...


,reports,age,income,owner,selfemp,dependents,months
0,0,33.00000,4.5200,1,0,3,54
1,0,33.00000,2.4200,0,0,3,34
2,0,33.00000,4.5000,1,0,4,58
3,0,33.00000,2.5400,0,0,0,25
4,0,33.00000,9.7867,1,0,2,64
...,...,...,...,...,...,...,...
1314,0,33.00000,4.5660,1,0,0,94
1315,5,23.91667,3.1920,0,0,3,12
1316,0,33.00000,4.6000,1,0,2,1
1317,0,33.00000,3.7000,0,1,0,60


In [ ]:
pipe = sk.pipeline.Pipeline(steps=[
    ("am", AgeMaxer()),
    ("clf", sk.ensemble.RandomForestClassifier(n_estimators=100, random_state=42)),
])

scores = sk.model_selection.cross_val_score(pipe, X, y, scoring="accuracy", cv=10, n_jobs=-1)
print("scores mean:", scores.mean())